# Wikipedia Sentiment Analysis

**Goal:** Given a Wikipedia article, compute sentiment statistics, visualize how sentiment flows through the text, compare articles, and map sentiment across a Wikipedia knowledge graph.

**Pipeline:**
1. Fetch article text via the Wikipedia API (no dataset required)
2. Run DistilBERT sentiment classification on text chunks
3. Visualize progression, sections, article comparisons
4. Build a knowledge graph and propagate sentiment across linked articles
5. Generate a word cloud

**Recommended runtime:** GPU (Runtime → Change runtime type → T4 GPU in Colab)

In [ ]:
# ── Colab setup ──────────────────────────────────────────────────────────────
# Run this cell once. It clones the repo and installs dependencies.
import os

if not os.path.exists('src'):
    !git clone https://github.com/RazaTypesCode/sentiment-analysis-wikipedia.git
    %cd sentiment-analysis-wikipedia

!pip install -q -r requirements.txt

In [ ]:
import sys
sys.path.insert(0, '.')

import matplotlib.pyplot as plt

from src.wiki_loader import get_article, get_sections, build_graph_data
from src.sentiment import (
    analyze, analyze_sections, average_score, section_averages
)
from src.visualization import (
    plot_sentiment_progression,
    plot_section_sentiment,
    plot_article_comparison,
    plot_knowledge_graph,
    plot_wordcloud,
)

print('All imports OK')

## 1. Single Article — Sentiment Progression

The article is split into 400-character chunks. Each chunk is classified as POSITIVE or NEGATIVE by DistilBERT. The score is mapped to `[-1, 1]` and plotted in order, so you can see how sentiment shifts as the article progresses.

In [ ]:
ARTICLE = 'Quantum mechanics'   # ← change this to any Wikipedia article title

print(f'Fetching: {ARTICLE}')
text = get_article(ARTICLE)
print(f'Article length: {len(text):,} characters')

print('Running sentiment analysis...')
results = analyze(text)

avg = average_score(results)
print(f'Chunks analyzed : {len(results)}')
print(f'Average sentiment: {avg:+.3f}  ({"positive" if avg > 0 else "negative"})')

In [ ]:
fig = plot_sentiment_progression(results, title=f'Sentiment Progression — {ARTICLE}')
plt.show()

## 2. Section-Wise Sentiment

Wikipedia articles are divided into named sections. Here we analyze each section independently and compare them on a horizontal bar chart.

In [ ]:
print(f'Fetching sections for: {ARTICLE}')
sections = get_sections(ARTICLE)
print(f'Found {len(sections)} sections')

print('Analyzing sections...')
sec_results = analyze_sections(sections)
sec_avgs = section_averages(sec_results)

for name, score in sorted(sec_avgs.items(), key=lambda x: -x[1]):
    bar = '█' * int(abs(score) * 20)
    sign = '+' if score > 0 else '-'
    print(f'  {sign}{abs(score):.2f}  {bar:20s}  {name}')

In [ ]:
fig = plot_section_sentiment(sec_avgs, title=f'Section Sentiment — {ARTICLE}')
plt.show()

## 3. Comparing Articles

Compute the average sentiment score for several articles and compare them side-by-side. This reveals which topics Wikipedia tends to frame more positively or negatively.

In [ ]:
ARTICLES = [
    'Black hole',
    'Climate change',
    'Quantum mechanics',
    'Michael Jackson',
    'World War II',
    'Artificial intelligence',
]

article_scores = {}
for title in ARTICLES:
    print(f'Analyzing: {title}...')
    t = get_article(title)
    r = analyze(t)
    article_scores[title] = average_score(r)
    print(f'  → {article_scores[title]:+.3f}')

fig = plot_article_comparison(article_scores)
plt.show()

## 4. Sentiment Propagation on the Wikipedia Knowledge Graph

Starting from a seed article, we follow Wikipedia hyperlinks to build a directed graph. We then run sentiment analysis on every article in the graph and visualize the result as a network where:

- **Node color** → red (negative) to green (positive)
- **Node size** → magnitude of sentiment (more extreme = larger)
- **Edges** → hyperlink direction from one article to another

> **Tip:** Keep `MAX_LINKS` small (≤ 10) and `DEPTH = 1` for a fast run. Increase depth only with a GPU.

In [ ]:
SEED      = 'Quantum mechanics'
DEPTH     = 1
MAX_LINKS = 8

print(f'Building graph from "{SEED}" (depth={DEPTH}, max_links={MAX_LINKS})')
nodes, edges = build_graph_data(SEED, depth=DEPTH, max_links=MAX_LINKS)
print(f'Graph: {len(nodes)} nodes, {len(edges)} edges')

In [ ]:
node_scores = {}
for i, title in enumerate(nodes):
    print(f'[{i+1}/{len(nodes)}] {title}')
    try:
        t = get_article(title)
        r = analyze(t)
        node_scores[title] = average_score(r)
    except Exception as exc:
        print(f'    skipped: {exc}')
        node_scores[title] = 0.0

print('\nMost positive:')
for t, s in sorted(node_scores.items(), key=lambda x: -x[1])[:3]:
    print(f'  {s:+.3f}  {t}')

print('\nMost negative:')
for t, s in sorted(node_scores.items(), key=lambda x: x[1])[:3]:
    print(f'  {s:+.3f}  {t}')

In [ ]:
fig = plot_knowledge_graph(
    nodes, edges, node_scores,
    title=f'Sentiment Knowledge Graph — {SEED}'
)
plt.show()

## 5. Word Cloud

In [ ]:
fig = plot_wordcloud(text, title=f'Word Cloud — {ARTICLE}')
plt.show()